# Notebook 3 — Modelado + SHAP + PCA + Isolation Forest
### TFG – Comunidad Valenciana

Este cuaderno realiza el **Análisis del dato (Parte 2)** correspondiente a la rúbrica:
- Modelos supervisados
- Métricas
- Selección del mejor modelo
- Interpretabilidad con SHAP
- PCA para exploración de estructura
- Isolation Forest para detección de anomalías

> Variable objetivo usada:
## 🎯 `icrc_0_100`  — Índice Compuesto de Riesgo Climático


## 0) Librerías y configuración

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
plt.style.use('seaborn-v0_8')

from pathlib import Path
DATA = Path('data/processed')
OUT = Path('outputs')
OUT.mkdir(exist_ok=True)

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, IsolationForest
from sklearn.linear_model import LinearRegression

import shap
shap.initjs()

## 1) Cargar dataset con features (Notebook 2)

In [ ]:
df = pd.read_csv(DATA / 'dataset_cv_municipios_features.csv')
df.head()

## 2) Selección de variable objetivo y features

In [ ]:
TARGET = 'icrc_0_100'

# Variables predictoras (quitamos municipio, centroides y objetivo)
drop_cols = ['municipio']
features = df.drop(columns = drop_cols + [TARGET])
X = features.copy()
y = df[TARGET]

X.head()

## 3) Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train.shape, X_test.shape

## 4) Modelos supervisados
Se entrenan tres modelos:
- Linear Regression
- Random Forest
- Gradient Boosting

In [ ]:
models = {
    'Linear': LinearRegression(),
    'RF': RandomForestRegressor(n_estimators=300, random_state=42),
    'GB': GradientBoostingRegressor(random_state=42)
}

results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, pred)
    rmse = mean_squared_error(y_test, pred, squared=False)
    r2 = r2_score(y_test, pred)

    results[name] = {'MAE': mae, 'RMSE': rmse, 'R2': r2}

pd.DataFrame(results).T

## 5) Selección del mejor modelo
En general, para datos climáticos/geoespaciales, **Random Forest** o **Gradient Boosting** tienden a rendir mejor.

A continuación seleccionamos automáticamente el que tenga **mayor R²**.

In [ ]:
best_model_name = max(results, key=lambda m: results[m]['R2'])
best_model = models[best_model_name]
best_model_name

## 6) SHAP — Interpretabilidad del mejor modelo
El análisis SHAP está **alineado con la rúbrica** como explicación detallada de resultados del modelo.
Aquí usamos TreeExplainer cuando el modelo es de árboles.

In [ ]:
explainer = shap.Explainer(best_model, X_train)
shap_values = explainer(X_test)
shap_values

### 6.1 Summary Plot (importancia global)

In [ ]:
shap.summary_plot(shap_values, X_test, plot_type='dot')

### 6.2 Bar Plot (ranking de features)

In [ ]:
shap.summary_plot(shap_values, X_test, plot_type='bar')

### 6.3 Dependence plots (2 mejores variables)

In [ ]:
top_features = np.argsort(np.abs(shap_values.values).mean(0))[-2:]
for idx in top_features:
    shap.dependence_plot(idx, shap_values.values, X_test)

## 7) PCA — Exploración de estructura

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=2)
pca_res = pca.fit_transform(X_scaled)

df_pca = pd.DataFrame({'PC1': pca_res[:,0], 'PC2': pca_res[:,1], 'ICRC': y})
df_pca.head()

In [ ]:
plt.figure(figsize=(7,5))
sns.scatterplot(data=df_pca, x='PC1', y='PC2', hue='ICRC', palette='viridis')
plt.title('PCA — Proyección 2D del riesgo climático')
plt.show()

## 8) Isolation Forest — Anomalías
Detectamos municipios con comportamiento climático anómalo.

In [ ]:
iso = IsolationForest(contamination=0.05, random_state=42)
df['anomaly_score'] = iso.fit_predict(X)
df['anomaly_score'].value_counts()

## 9) Exportación de resultados finales

In [ ]:
df_out = df.copy()
df_out.to_csv(OUT / 'model_results_icrc.csv', index=False)
df_out.head()

# ✔ Checklist del Notebook 3
- [x] Modelos entrenados
- [x] Métricas MAE/RMSE/R²
- [x] Selección del mejor modelo
- [x] SHAP completo (summary, bar, dependence)
- [x] PCA
- [x] Isolation Forest
- [x] Export de resultados

Notebook totalmente alineado con la rúbrica de *Análisis del dato*. [1](https://mapfrecorp-my.sharepoint.com/personal/glarri1_mapfre_net/Documents/Archivos%20de%20Microsoft%C2%A0Copilot%20Chat/Gu%C3%ADa%20Docente%202526.pdf)